In [ ]:
import pandas as pd
import os, glob
candidates = [
    'Datas/QVI_data.csv', 'Datas\\QVI_data.csv', './Datas/QVI_data.csv', '../Datas/QVI_data.csv',
    'Data/QVI_data.csv', '../Data/QVI_data.csv', 'Datasets/QVI_data.csv', '../Datasets/QVI_data.csv',
    'QVI_data.csv'
]

qvi_path = next((p for p in candidates if os.path.exists(p)), None)
if qvi_path is None:
    matches = glob.glob('**/QVI_data.csv', recursive=True)
    qvi_path = matches[0] if matches else None

if qvi_path is None:
    raise FileNotFoundError("QVI_data.csv not found. Put the file in the notebook directory or update the path.")

qvi = pd.read_csv(qvi_path)

FileNotFoundError: [Errno 2] No such file or directory: 'Datas\\QVI_data.csv'

In [ ]:
qvi = qvi[~qvi['PROD_NAME'].str.contains('SALSA',case=False)]

In [ ]:
qvi.drop_duplicates(inplace=True)

In [ ]:
qvi['DATE'] = pd.to_datetime(qvi['DATE'])

qvi['Month_id']=qvi['DATE'].dt.to_period('M')
grouped = qvi.groupby(['STORE_NBR'])['Month_id'].nunique()
not_all_tns = grouped[grouped!=12].index
all_tns = grouped[grouped==12].index

qvi.drop(qvi[qvi['STORE_NBR'].isin(not_all_tns)].index, inplace=True)

print(not_all_tns)
print(all_tns)

In [ ]:
qvi.head()

In [ ]:
# for analysis we store the matrics in single data frame 
# easy for correlation and plotting

In [ ]:
Measure_over_time = qvi.groupby(['STORE_NBR','Month_id']).agg({'TOT_SALES':'sum',
                                                               'LYLTY_CARD_NBR':pd.Series.nunique,
                                                               'TXN_ID': 'count',
                                                               'PROD_QTY': 'sum'}).reset_index()

Measure_over_time = Measure_over_time.rename(columns={'LYLTY_CARD_NBR':'NUM_CUS',
                                                      'TXN_ID':'NUM_TNS'})

Measure_over_time

In [ ]:
type(Measure_over_time)

In [ ]:
Measure_over_time ['TXN_PER_CUS'] = (Measure_over_time['NUM_TNS'] / Measure_over_time['NUM_CUS'])

Measure_over_time

In [ ]:
Measure_over_time ['AVG_PRICE'] = (Measure_over_time['TOT_SALES'] / Measure_over_time['PROD_QTY'])

Measure_over_time

In [ ]:
# it have measures before implementing trial store
preTrailMeasures = Measure_over_time[(Measure_over_time['Month_id'] < pd.Period('2019-02',freq='M'))]
preTrailMeasures

In [ ]:
# we create a correlation func with 3 input parameters [inputTable, metricCol, storecomparision]
# inputTable - contain store metrics
# metricCol - Which metric column we compare
# storecomparision - trial store number

# Correlation Score → Checks whether the trend/pattern is similar.
# Magnitude Score → Checks whether the actual values (sales, customers) are similar.

#  we create table with store-1, store-2, corr_measure, magnitude

In [ ]:
def find (input_table, metric_col,store_comparision) :
    
    pivot_table = input_table.pivot(index = 'Month_id',
                                    columns = 'STORE_NBR',
                                    values = metric_col)
    
    metric_table = pivot_table.corr()[store_comparision].reset_index()
    metric_table.columns = ['STORE-2','CORR_MEASURE']
    metric_table['STORE-1']= store_comparision
    metric_table = metric_table[['STORE-1','STORE-2','CORR_MEASURE']]
    
    trail_store = pivot_table[store_comparision]
    result=[]
    for i in pivot_table.columns :
        current_store = pivot_table[i]
        
        diff = abs(trail_store-current_store)
        
        avg_diff = diff.mean()
        
        result.append({'AVG_DIFF':avg_diff})
        
    df=pd.DataFrame(result)
        
    metric_table = metric_table.join(df)
    
    min_diff=metric_table['AVG_DIFF'].min()
    max_diff=metric_table['AVG_DIFF'].max()
    
    metric_table['MAG_SCORE'] = 1-((metric_table['AVG_DIFF']-min_diff)/(max_diff-min_diff))
    metric_table['COMPOSITE_VAL'] = ((metric_table['CORR_MEASURE']+metric_table['MAG_SCORE'])/2)
    
    metric_table = metric_table[['STORE-1','STORE-2','COMPOSITE_VAL']]

    return metric_table

compo_sales_77 = find(preTrailMeasures, 'TOT_SALES',77)
compo_sales_77
    

In [ ]:
# calling the function to find corr for monthly no. of customer

compo_cus_77 = find(preTrailMeasures, 'NUM_CUS',77)
compo_cus_77

In [ ]:
#for the trial store 86,88 (call func to find composite value which is addition of correlation and magnitude) both sales and customer
compo_sales_86 = find(preTrailMeasures, 'TOT_SALES',86)
compo_cus_86 = find(preTrailMeasures, 'NUM_CUS',86)

compo_sales_88 = find(preTrailMeasures, 'TOT_SALES',88)
compo_cus_88 = find(preTrailMeasures, 'NUM_CUS',88)

In [ ]:
# we merge the sales and customer calculation for the store trial store

final_df_77 = pd.merge(compo_cus_77,compo_cus_77,on=['STORE-1','STORE-2'])
final_df_77=final_df_77.rename(columns={'COMPOSITE_VAL_x':'SALES_SCORE',
                                  'COMPOSITE_VAL_y':'CUSTOMER_SCORE'})

final_df_86 = pd.merge(compo_cus_86,compo_cus_86,on=['STORE-1','STORE-2'])
final_df_86=final_df_86.rename(columns={'COMPOSITE_VAL_x':'SALES_SCORE',
                                  'COMPOSITE_VAL_y':'CUSTOMER_SCORE'})

final_df_88 = pd.merge(compo_cus_88,compo_cus_88,on=['STORE-1','STORE-2'])
final_df_88=final_df_88.rename(columns={'COMPOSITE_VAL_x':'SALES_SCORE',
                                  'COMPOSITE_VAL_y':'CUSTOMER_SCORE'})

# final control score is calculated by ADDING the sales and customer COMPOSITE value

final_df_77['FinalControlScore_77'] = (final_df_77['SALES_SCORE']* 0.5 )+(final_df_77['CUSTOMER_SCORE']*0.5)
print(final_df_77)

final_df_86['FinalControlScore_86'] = (final_df_86['SALES_SCORE']* 0.5 )+(final_df_86['CUSTOMER_SCORE']*0.5)
print(final_df_86)

final_df_88['FinalControlScore_88'] = (final_df_88['SALES_SCORE']* 0.5 )+(final_df_88['CUSTOMER_SCORE']*0.5)
final_df_88

In [ ]:
# to find the best control store we dicard the trial store and list the val in descending order and take the top value as control store
final_df_77=final_df_77[final_df_77['STORE-2']!=77]
final_df_77=final_df_77.sort_values(by='FinalControlScore_77',ascending=False)
print(final_df_77)

final_df_86=final_df_86[final_df_86['STORE-2']!=86]
final_df_86=final_df_86.sort_values(by='FinalControlScore_86',ascending=False)
print(final_df_86)

final_df_88=final_df_88[final_df_88['STORE-2']!=88]
final_df_88=final_df_88.sort_values(by='FinalControlScore_88',ascending=False)
final_df_88

In [ ]:
# best CONTROL STORE for the trial stores 
print(final_df_77['STORE-2'].head(1))

print('\n',final_df_86['STORE-2'].head(1))

final_df_88['STORE-2'].head(1)



In [ ]:
type(Measure_over_time)

In [ ]:
# we create the store_type col for the trial store 77
Measure_over_time['STORE_TYPE'] = 'other'
Measure_over_time.loc[Measure_over_time['STORE_NBR']==77,'STORE_TYPE'] = 'Trial'

Measure_over_time.loc[Measure_over_time['STORE_NBR']==233,'STORE_TYPE'] = 'Control'

Measure_over_time = Measure_over_time[['STORE_NBR','Month_id','TOT_SALES','STORE_TYPE','NUM_CUS']]

Measure_over_time

In [ ]:
Measure_over_time =Measure_over_time.rename(columns={'STORE_TYPE':'STORE_TYPE_77'})

In [ ]:
# created the store type col for the trial store 86 and 88
Measure_over_time['STORE_TYPE_86'] = 'other'
Measure_over_time['STORE_TYPE_88'] = 'other'

Measure_over_time.loc[Measure_over_time['STORE_NBR']==86,'STORE_TYPE_86'] = 'Trial'
Measure_over_time.loc[Measure_over_time['STORE_NBR']==155,'STORE_TYPE_86'] = 'Control'

Measure_over_time.loc[Measure_over_time['STORE_NBR']==88,'STORE_TYPE_88'] = 'Trial'
Measure_over_time.loc[Measure_over_time['STORE_NBR']==237,'STORE_TYPE_88'] = 'Control'

Measure_over_time = Measure_over_time[['STORE_NBR','Month_id','TOT_SALES','STORE_TYPE_77','STORE_TYPE_86','STORE_TYPE_88','NUM_CUS']]

Measure_over_time

In [ ]:
Measure_over_time

In [ ]:
# group the store type and dates then finding mean for total sales

Measure_over_time_77=Measure_over_time.groupby(['Month_id','STORE_TYPE_77']) ['TOT_SALES'].mean().reset_index()


Measure_over_time_86=Measure_over_time.groupby(['Month_id','STORE_TYPE_86']) ['TOT_SALES'].mean().reset_index()


Measure_over_time_88=Measure_over_time.groupby(['Month_id','STORE_TYPE_88']) ['TOT_SALES'].mean().reset_index()


In [ ]:
import matplotlib.pyplot as plt
def plotting (table,ST):
    plt.figure(figsize=(15,4))
    df=table[table[ST].isin(['Trial','Control'])]
    for storeT in df[ST].unique():
    
        temp=df[df[ST]==storeT]
    
        plt.plot(temp['Month_id'].astype(str),temp['TOT_SALES'],marker='o',label=storeT)
    
    plt.axvline(x='2019-02',linestyle='--',label='Trial Store',color='red')

    plt.title("Average Sales Before and After Trial")
    plt.xlabel("Month")
    plt.ylabel("Average Total Sales")
    plt.legend()
    plt.xticks(rotation=45)
    plt.show()
    

In [ ]:
plotting(Measure_over_time_77,'STORE_TYPE_77')

In [ ]:
plotting(Measure_over_time_86,'STORE_TYPE_86')

In [ ]:
plotting(Measure_over_time_88,'STORE_TYPE_88')

In [ ]:
# pre trail months  - 1 (before implementing trial store we have 6 months of data)
Dof = 7 -1

In [ ]:
from scipy.stats import t
critical_t = t.ppf(0.95,6)
critical_t

In [ ]:
def insight(t,c,measure):
    # scale the control store to the trial store baseline 

    scalefactor = preTrailMeasures.loc[(preTrailMeasures['STORE_NBR']==t) & (preTrailMeasures['Month_id'] < pd.Period('2019-02',freq='M')),measure].sum() / preTrailMeasures.loc[(preTrailMeasures['STORE_NBR']==c) & (preTrailMeasures['Month_id'] < pd.Period('2019-02',freq='M')),measure].sum()
    measurescaledsales=Measure_over_time.copy()
    trial = measurescaledsales.loc[measurescaledsales['STORE_NBR']==t,['STORE_NBR','Month_id', measure]].copy()
    control = measurescaledsales.loc[measurescaledsales['STORE_NBR']==c,['STORE_NBR','Month_id', measure]].copy()
    
    control['controlsales'] = control[measure]*scalefactor
    
    PD = trial.merge(control[['controlsales','Month_id']], on='Month_id',how ='inner')
    
    PD['perc_diff'] = abs((PD[measure] - PD['controlsales']) / PD['controlsales']) * 100
    
    SD= PD.loc[PD['Month_id'] < pd.Period('2019-02',freq='M'),'perc_diff'].std() 
    
    ''' Test Statistic= x − μ / Standard Deviation
        x = Observed value (percentage difference)
        μ (mu) = Expected value under the null hypothesis (Zero)
        SD = Standard deviation '''
    
    PD['Ttest'] = PD['perc_diff'] / SD
    relative_sd = SD / 100
    
    Trial_assessment = PD.loc[(PD['Month_id'] >= pd.Period('2019-02',freq='M')) & (PD['Month_id'] < pd.Period('2019-05',freq='M'))].copy()
    
    # confidence level range is where the normal distribution is expected to fall 95% of the time.
    Trial_assessment['lower_confidence'] = (Trial_assessment['controlsales'] * (1 - 2 * relative_sd))
    Trial_assessment['upper_confidence'] = (Trial_assessment['controlsales'] * (1 + 2 * relative_sd))
    
    
    Trial_assessment = Trial_assessment[['Month_id', measure, 'controlsales', 'lower_confidence', 'upper_confidence']]

    plt.figure(figsize=(15,5))
    plt.plot(Trial_assessment['Month_id'].astype(str),Trial_assessment[measure],marker='o',color='#18966f',label = 'Trial Store Sales')
    plt.plot(Trial_assessment['Month_id'].astype(str),Trial_assessment['controlsales'],marker='o',color='#20baba',label = 'Control Store Sales')
    plt.plot(Trial_assessment['Month_id'].astype(str),Trial_assessment['lower_confidence'],linestyle='--',color='gray',label = 'Confidence Interval')
    plt.plot(Trial_assessment['Month_id'].astype(str),Trial_assessment['upper_confidence'],linestyle='--',color='gray')

    plt.title('IS the Trial Store Sales Significantly Different from Control Store?')
    plt.xlabel('Month')
    plt.ylabel('Sales')
    plt.legend()

    return  PD, SD

In [ ]:
insight(77,233,'TOT_SALES')

In [ ]:
insight(86,155,'TOT_SALES')

In [ ]:
insight(88,237,'TOT_SALES')

In [ ]:
insight(77,233,'NUM_CUS')

In [ ]:
insight(86,155,'NUM_CUS')

In [ ]:
insight(88,237,'NUM_CUS')

In [ ]:
# Based on these plots 

# we can conclude that trial store has more sales and customers in Feb month ,top in march and decreased in april month

# from the plots,the store 77 and 88 gives positive results whereas in store 86 it doesn't seem better growth

# Based on analysis, i recommend to implement the trial store to increase the sales and customers.